# Incidentes e evidências de RCA do dump Argus

## Escopo, tempo e limites metodológicos

`opened_at` é a hora **ticket-open**, não o início do incidente. A linha do
tempo usa uma banda visual de **40 minutos** de incerteza sem alterar o
timestamp armazenado. A ausência de ticket não é rótulo negativo de ausência
de problema: incidentes podem não ser registrados ou ser registrados tarde.

O escopo padrão `video` usa somente `incidents`. `platform` é um universo de
times vizinhos e só deve ser consultado explicitamente, sem misturar ambos na
avaliação.

In [6]:
from dump_explorer.charts import (
    attribution_stack,
    incident_timeline,
    provenance_bar,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.rca import (
    build_incident_window,
    load_dashboard_artifact,
    load_error_attribution,
    load_incidents,
    load_labels,
)

# derive a safe UTC epoch-ms window from signal_bucket observed bounds
from dump_explorer.quality import load_source_coverage
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

# Obter janela segura a partir da cobertura de dados
s = load_source_coverage(engine)
if s.empty or s['first_ts'].dropna().empty:
    print("⚠️  source_coverage vazio. Usando valores padrão.")
    start_ms = 0
    end_ms = 86_400_000
else:
    start_ms = int(s['first_ts'].dropna().min().value // 10**6)
    end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
    print(f"Intervalo de dados: {pd.to_datetime(start_ms, unit='ms', utc=True)} a {pd.to_datetime(end_ms-1, unit='ms', utc=True)}")

Intervalo de dados: 2026-07-10 18:45:00+00:00 a 2026-08-05 23:45:00+00:00


## 1. Tickets estruturados e cobertura de telemetria

O leitor só expõe prioridade, estado, duração registrada e componentes
estruturados; texto livre sanitizado não é exibido. Compare a cobertura de
ticket-open com a cobertura de telemetria para investigar lacunas, não para
inferir que uma das fontes causou a outra.

In [7]:
incidents = load_incidents(engine, start_ms, end_ms, scope="video")
attribution = load_error_attribution(engine, start_ms, end_ms)

display(incidents[["opened_at", "priority", "state", "duration_ms"]])
display(
    {
        "ticket_open_rows": len(incidents),
        "telemetry_rows": len(attribution),
        "ticket_open_min": incidents["opened_at"].min() if not incidents.empty else None,
        "telemetry_min": attribution["timestamp"].min() if not attribution.empty else None,
    }
)

,opened_at,priority,state,duration_ms
0,2026-07-10 20:27:38+00:00,P5,Finalizado - Cancelado,178415000.0
1,2026-07-10 21:05:48+00:00,P4,Aguardando Atendimento,NaN
2,2026-07-11 07:51:26+00:00,P4,Aguardando Atendimento,NaN
3,2026-07-11 21:27:16+00:00,P4,Finalizado - Solucionado,3159000.0
4,2026-07-11 21:43:48+00:00,P4,Aguardando Atendimento,NaN
...,...,...,...,...
238,2026-08-05 17:54:46+00:00,P5,Solucionado,3389000.0
239,2026-08-05 18:12:50+00:00,P5,Pendente,NaN
240,2026-08-05 21:06:36+00:00,P4,Solucionado,2170000.0
241,2026-08-05 22:00:54+00:00,P5,Pendente,NaN


{'ticket_open_rows': 243,
 'telemetry_rows': 65673,
 'ticket_open_min': Timestamp('2026-07-10 20:27:38+0000', tz='UTC'),
 'telemetry_min': Timestamp('2026-07-22 00:00:00+0000', tz='UTC')}

## 2. Janela de evidência

Os offsets permitidos são 15m, 30m, 1h, 2h e 6h. O contexto reúne ticket,
deploy e atribuição de erros no tempo; a sobreposição é evidência para
investigação, não prova de causalidade.

In [8]:
if not incidents.empty:
    context = build_incident_window(engine, incidents.iloc[0], before="1h", after="1h")
    incident_timeline(context.incident.to_frame().T, context.deploys, context.signals).show()
    attribution_stack(context.attribution).show()
    display(context.labels)
else:
    print("Sem ticket na janela: isso não é uma evidência negativa de incidente.")

,tier,kind,ts_start,ts_end,severity,region,services,services_parse_error,metrics_cited,metrics_cited_parse_error,provenance,circular,evaluation_eligible


## 3. Golden set e proveniência

Por padrão, a avaliação usa apenas `rca-confirmed` e `team-confirmed`.
`detector-derived` pode aparecer na visualização, mas é marcado como circular:
ele mede concordância com o detector que o criou, não acurácia independente.

In [9]:
confirmed_labels = load_labels(engine)
display(confirmed_labels)
provenance_bar(confirmed_labels).show()

# Somente para inspeção visual; não use este conjunto como ground truth.
detector_display = load_labels(engine, provenances=["detector-derived"])
provenance_bar(detector_display).show()

,tier,kind,ts_start,ts_end,severity,region,services,services_parse_error,metrics_cited,metrics_cited_parse_error,provenance,circular,evaluation_eligible
0,1,incident,2026-04-16 21:13:00+00:00,2026-04-16 21:48:00+00:00,P1,None,"[videos-api, channels-api, quicksilver, jarvis...",False,"{'videos_api_fetcherror_30min': 7944, 'videos_...",False,rca-confirmed,False,True
1,2,load_test,2026-04-29 21:00:00+00:00,2026-04-30 03:00:00+00:00,None,None,[*],False,"{'newrelic_err_pct_pico': 0.51, 'newrelic_err_...",False,team-confirmed,False,True
2,2,incident,2026-05-06 20:00:00+00:00,2026-05-06 21:30:00+00:00,?,None,"[jarvis-graph/Broadcasts, jarvis-graph/Details...",False,{'apollo_DetailsSoccerMatchEvent_err_pct': 28....,False,team-confirmed,False,True
3,2,load_test,2026-05-20 03:00:00+00:00,2026-05-20 09:00:00+00:00,None,None,[*],False,"{'buckets_internal_utc': ['04:00', '05:00', '0...",False,team-confirmed,False,True
4,2,load_test,2026-05-26 03:00:00+00:00,2026-05-26 10:00:00+00:00,None,None,[*],False,"{'buckets_internal_utc': ['02:00', '04:00', '0...",False,team-confirmed,False,True
5,1,incident,2026-07-05 20:00:00+00:00,2026-07-05 23:00:00+00:00,P2,BR,"[globo-fifa26, cdn_live, playback-api, delivery]",False,"{'concurrent_peak': 2429007, 'delivery_session...",False,team-confirmed,False,True


## 4. Comparação opcional com artefatos Argus

Um artefato só é lido por nome exato e seu JSON precisa ser um objeto. Esta
comparação é **opcional**: associe-o a um rótulo humano apenas se ambos trouxerem
um identificador compartilhado explícito. Não faça associação por título,
texto parecido ou proximidade temporal.

In [10]:
artifact_name = "incidents"  # nome exato, altere conscientemente
artifact = load_dashboard_artifact(engine, artifact_name)
display(artifact)

# Sem identificador compartilhado explícito, mantenha artefato e rótulos separados.

{'window': {'start': '2026-07-23T21:15:05+00:00',
  'end': '2026-08-06T21:15:05+00:00'},
 'z_threshold': 3.5,
 'min_sources': 2,
 'positive_only': True,
 'ranking_min_score': 1000.0,
 'hours': [{'ts': '2026-07-29T00:00:00+00:00',
   'score': 12623.31,
   'anomalies': [{'source': 'origin_live_errors',
     'entity': 'globo-fab',
     'metric': 'err_rate',
     'ts': '2026-07-29T00:00:00+00:00',
     'value': 2.4333,
     'baseline': 0.0,
     'z': 50.0,
     'detector': 'stat',
     'fold': 243.33,
     'mag': 2.39,
     'weighted_z': 169.31,
     'series': [{'t': 1785272518949, 'v': 0.0},
      {'t': 1785272818950, 'v': 0.0},
      {'t': 1785272827118, 'v': 0.0},
      {'t': 1785273118949, 'v': 0.0},
      {'t': 1785273127119, 'v': 0.0},
      {'t': 1785273418949, 'v': 0.0},
      {'t': 1785273427118, 'v': 0.0},
      {'t': 1785273427810, 'v': 0.0},
      {'t': 1785273727118, 'v': 0.0},
      {'t': 1785273727809, 'v': 0.0},
      {'t': 1785274027119, 'v': 0.0},
      {'t': 178527402780